In [0]:
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "src")))

import dlt
from pyspark.sql.functions import col
from cdc_pipeline.config import LANDING_PATH


@dlt.table(name="bronze_orders_cdc",
           comment="Raw CDC changelog event")
def bronze_orders_cdc():
    return (
        spark.readStream.format("cloudFiles") \
        .option("cloudFiles.format", "json") \
            .load(LANDING_PATH)
    )


dlt.create_streaming_table("current_orders")

dlt.applychange(
   target = "current_orders", 
   source = "bronze_orders_cdc",
   keys = ["order_id"],
   sequence_by = col("seq"),
   apply_as_deletes=col("op") == "DELETE",
   except_column_list=["op", "seq"],
   stored_as_scd_type=1,
)